In [1]:
import torch
import pandas as pd
import time
from tqdm import tqdm
from unsloth import FastLanguageModel

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
print("PyTorch version:", torch.__version__)
print("Torch Built With CUDA:", torch.backends.cuda.is_built())
print("Current CUDA Device:", torch.cuda.current_device())
print("CUDA Version (Torch):", torch.version.cuda)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/sdonev/data/conda/envs/unsloth_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Unsloth: Failed to patch SmolVLMForConditionalGeneration forward function.
🦥 Unsloth Zoo will now patch everything to make training faster!
CUDA available: True
GPU: NVIDIA H100 80GB HBM3
PyTorch version: 2.5.1
Torch Built With CUDA: True
Current CUDA Device: 0
CUDA Version (Torch): 12.1


# Load Model

In [2]:
model_name="unsloth/Meta-Llama-3.1-8B"
model_name_str = model_name.lower().replace("-","_").replace("/","_")
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.


In [3]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=f"lora_model_{model_name_str}",        # points to the folder you saved above
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)
FastLanguageModel.for_inference(model)

==((====))==  Unsloth 2025.4.1: Fast Llama patching. Transformers: 4.51.3.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 1. Max memory: 79.209 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1. CUDA: 9.0. CUDA Toolkit: 12.1. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.28.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2025.4.1 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096, padding_idx=128004)
        (layers): ModuleList(
          (0): LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

In [4]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

In [7]:
inputs = tokenizer(
[
    alpaca_prompt.format(
        "What is the age of the used animals in this experiments? Return `AGE: <standardized age or descriptor>`", # instruction
        "Mice used in these studies were 2-3 month old males with a mixed C57BL/6 6 129S background", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)
tokenizer.batch_decode(outputs)

['<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nWhat is the age of the used animals in this experiments? Return `AGE: <standardized age or descriptor>`\n\n### Input:\nMice used in these studies were 2-3 month old males with a mixed C57BL/6 6 129S background\n\n### Response:\nAGE: 2-3 month old<|end_of_text|>']

# Load Data

In [8]:
bert_classified = pd.read_csv("./predictions/sentences_with_bert_age_predictions.csv")
bert_classified['doc_id_unique'] = bert_classified['PMID'].astype(str) + "_" + bert_classified['sentence_id'].astype(str).str.strip()

bert_classified.head()

,PMID,sentence_id,sent_txt,predicted_label,doc_id_unique
0,26277791,0,Animals Female 6- to 8-week-old C57BL/6 mice w...,1,26277791_0
1,26277791,4,Female 6- to 8-week-old C57BL/6 mice were obta...,1,26277791_4
2,26277791,8,Induction of EAE in Mice and Treatment Protoco...,1,26277791_8
3,26277791,13,Since no clinical symptoms are visible at scor...,0,26277791_13
4,26277791,15,Copaxone pretreatment was given for 10days bef...,0,26277791_15


In [9]:
bert_classified.predicted_label.value_counts()

predicted_label
0    14037
1     3462
Name: count, dtype: int64

In [10]:
bert_classified_pos = bert_classified[bert_classified["predicted_label"]==1]
bert_classified_pos.shape

(3462, 5)

# Inference

In [11]:
instructions_simpler = """### TASK ###

EXTRACT THE AGE OF ANIMALS MENTIONED IN THE TEXT.  
RETURN ONLY THE AGE IN A STANDARDIZED FORMAT.  
IF NO AGE IS GIVEN, RETURN `"AGE NOT SPECIFIED"`.

---

### HOW TO THINK (CHAIN OF THOUGHTS) ###

1. READ the sentence carefully.
2. FIND any age-related phrases (e.g., "8 weeks", "P30", "adult", "3 months").
3. LINK each age phrase to the animal(s) it describes.
4. STANDARDIZE the format:  
   - Use `<number> <unit>` (e.g., `8 weeks`, `3 months`)  
   - Keep terms like `adult`, `juvenile`, `neonatal` unchanged  
5. IF no age is mentioned, write: `"AGE NOT SPECIFIED"`

---

### OUTPUT FORMAT ###

- `AGE: <standardized age or descriptor>`

---

### EXAMPLES ###

#### INPUT 1 ####  
Gene deletion was induced in male and female 12- to 20-week-old mice.  
#### OUTPUT 1 ####  
AGE: 12-20 weeks

#### INPUT 2 ####  
Six adult male WAG/Rij rats were used.  
#### OUTPUT 2 ####  
AGE: adult

#### INPUT 3 ####  
Juvenile pigs (approximately 3 months old) were used.  
#### OUTPUT 3 ####  
AGE: 3 months

#### INPUT 4 ####  
For Experiment 2, male young (3-months-old) and aged (23-months-old) rats were used.  
#### OUTPUT 4 ####  
AGE: 3 months, 23 months

#### INPUT 5 ####  
Twenty Sprague Dawley rats were used; no details were provided on age.  
#### OUTPUT 4 ####  
AGE: AGE NOT SPECIFIED

---

### WHAT NOT TO DO ###

- DO NOT include the whole sentence in the output  
- DO NOT include weight, sex, or strain  
- DO NOT guess the age  
- DO NOT omit the unit (e.g., weeks/months)  
- DO NOT ignore terms like "adult", "neonatal", or "juvenile"  
- DO NOT return multiple values or unformatted strings

---
ONLY OUTPUT THE AGE USING THE FORMAT ABOVE. NOTHING ELSE.
"""

In [12]:
inputs = tokenizer(
[
    alpaca_prompt.format(
        instructions_simpler,        
        "Mice used in these studies were 2-3 month old males with a mixed C57BL/6 6 129S background", # instruction
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 500, use_cache = True)
tokenizer.batch_decode(outputs)

['<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\n### TASK ###\n\nEXTRACT THE AGE OF ANIMALS MENTIONED IN THE TEXT.  \nRETURN ONLY THE AGE IN A STANDARDIZED FORMAT.  \nIF NO AGE IS GIVEN, RETURN `"AGE NOT SPECIFIED"`.\n\n---\n\n### HOW TO THINK (CHAIN OF THOUGHTS) ###\n\n1. READ the sentence carefully.\n2. FIND any age-related phrases (e.g., "8 weeks", "P30", "adult", "3 months").\n3. LINK each age phrase to the animal(s) it describes.\n4. STANDARDIZE the format:  \n   - Use `<number> <unit>` (e.g., `8 weeks`, `3 months`)  \n   - Keep terms like `adult`, `juvenile`, `neonatal` unchanged  \n5. IF no age is mentioned, write: `"AGE NOT SPECIFIED"`\n\n---\n\n### OUTPUT FORMAT ###\n\n- `AGE: <standardized age or descriptor>`\n\n---\n\n### EXAMPLES ###\n\n#### INPUT 1 ####  \nGene deletion was induced in male and female 12- to 20-week-old mice.  

In [13]:
def extract_animal_age_unsloth(text, model, tokenizer, instructions):
    inputs = tokenizer(
        [
            alpaca_prompt.format(
                instructions,        
                text, # instruction
                "", # output - leave this blank for generation!
            )
        ], return_tensors = "pt").to("cuda")
        
    outputs = model.generate(**inputs, max_new_tokens = 500, use_cache = True)
    res = tokenizer.batch_decode(outputs)
    return res[0].split("Response:")[1].replace("<|end_of_text|>","").strip()

In [14]:
def run_predictions_and_collect_unsloth(df, text_col, model, tokenizer, instructions, output_path=None):
    predictions = []

    for i, row in tqdm(df.iterrows(), total=len(df)):
        pmid = row["PMID"]
        doc_id = row["doc_id_unique"]
        text = row[text_col]

        try:
            age_prediction = extract_animal_age_unsloth(text, model, tokenizer, instructions)
        except Exception as e:
            age_prediction = f"ERROR: {str(e)}"

        prediction_row = {
            "pmid": pmid,
            "doc_id_unique": doc_id,
            "ent_text": text,
            "age_prediction": age_prediction
        }

        predictions.append(prediction_row)

        if output_path:
            # Append row directly to file (write header only on first row)
            pd.DataFrame([prediction_row]).to_csv(
                output_path, mode='a', header=(i == 0), index=False
            )

    return pd.DataFrame(predictions)

In [ ]:
start_time = time.time()

predictions = run_predictions_and_collect_unsloth(
    bert_classified_pos,
    "sent_txt",
    model,
    tokenizer,
    instructions_simpler, #prompt_full_sentences, #instructions_simpler,
    output_path=f"predictions/age_{model_name_str}.csv"
)

end_time = time.time()
elapsed = end_time - start_time
print(f"\nCompleted predictions in {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")

  0%|          | 11/3462 [00:02<10:15,  5.61it/s]